# NexusMD — molecular dynamics of a docked pose

This notebook simulates the complex you exported and analyses the trajectory.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Without a GPU this
will run roughly 50x slower and you should use `--ns 0.2` just to check the pipeline.


## 1. Install the toolchain

`openff-toolkit` has no pip wheel, so this goes through conda. Takes about five
minutes and restarts the kernel once — that restart is expected.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()          # kernel restarts here; run the next cell afterwards


In [ ]:
!mamba install -y -q -c conda-forge \
    openmm openff-toolkit openmmforcefields pdbfixer \
    rdkit mdanalysis prolif matplotlib pandas

import openmm
from openmm import Platform
print("OpenMM", openmm.version.version)
print("platforms:", [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())])
assert "CUDA" in [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())], \
    "No CUDA platform — switch the runtime to a GPU, or run with --ns 0.2 on CPU."


## 2. Upload the package

Upload the zip NexusMD gave you, or the individual files
(`protein.pdb`, `ligand.sdf`, `run_md.py`, `analyse.py`).

In [ ]:
from google.colab import files
import zipfile, glob, os

up = files.upload()
for name in up:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            z.extractall(".")
        print("extracted", name)

# the package may unzip into a subfolder; move its contents up
for p in glob.glob("*/run_md.py"):
    d = os.path.dirname(p)
    for f in glob.glob(d + "/*"):
        os.replace(f, os.path.basename(f))
print(sorted(os.listdir(".")))


## 3. Simulate

`--ns 10` is a reasonable first run. Start with `--ns 1` if you want to see the whole
pipeline finish before committing hours to it.

In [ ]:
!python run_md.py --ns 10


## 4. Analyse

Produces `analysis/` with RMSD, RMSF, radius of gyration, ligand displacement,
hydrogen bond occupancy, the interaction fingerprint and a written summary.

In [ ]:
!python analyse.py


In [ ]:
from IPython.display import Image, Markdown, display
import os

print(open("analysis/summary.md").read())

for png in ["rmsd.png", "ligand_displacement.png", "rmsf.png",
            "hbond_occupancy.png", "interaction_occupancy.png",
            "interaction_timeline.png", "rgyr.png"]:
    path = os.path.join("analysis", png)
    if os.path.exists(path):
        display(Markdown(f"### {png}"))
        display(Image(path))


## 5. Download the results

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("md_results", "zip", ".", "analysis")
files.download("md_results.zip")


---

**Interpreting it.** Ligand RMSD settling below ~2 Å means the docked pose held.
A steady climb with centre-of-mass drift past 5 Å means the ligand left the site and
the docking score was optimistic.

A contact present in 90% of frames is a binding determinant. One appearing in 15% is
noise. Ten nanoseconds tests pose stability, not affinity — and one trajectory is one
sample, so three runs with different seeds tell you much more than one long run.
